# Clinical Strand — Explore, Preprocess & Train

**MultimodalAI'26 Hackathon · 10 June 2026**

This notebook takes you from raw data to three saved models.
Once complete, launch the evaluation and reporting app:
```bash
cd ..          # back to clinical_strand/
streamlit run app.py
```

---
**Track role reminder — where to spend time after Step 8:**
- **The Explainer** → feature importance and calibration analysis
- **The Failure Hunter** → subgroup analysis and ablation experiments
- **The Gatekeeper** → validation summary table and the app's Report Builder

> **Prerequisite:** implement `src/evaluate.py` before running Step 5 onwards.
> The metrics functions are defined there. Without them, Steps 5–8 will error.

In [ ]:
# Uncomment if running in Google Colab
# !pip install lightgbm scikit-learn plotly -q
# import sys; sys.path.insert(0, '/content/clinical_strand')

In [ ]:
import sys, warnings
sys.path.insert(0, '..')   # makes models/ and src/ importable
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import joblib
from pathlib import Path

# Model A is provided complete.
# Uncomment Model B and C after you have implemented them in models/.
from models import ModelA
from models.model_b import ModelB
from models.model_c import ModelC

# Evaluation functions — implement these in src/evaluate.py first.
from src.evaluate import (
    compute_metrics,
    compute_subgroup_metrics,
    compute_calibration,
    roc_curve_data,
    equity_gap,
    age_band,
    save_metrics_csv,
)

RAW_DATA   = Path('../data/raw/icu_patients.csv')
PROC_DIR   = Path('../data/processed')
MODEL_DIR  = Path('../saved_models')
METRIC_DIR = Path('../saved_metrics')

for d in [PROC_DIR, MODEL_DIR, METRIC_DIR]:
    d.mkdir(exist_ok=True)

print('Setup complete.')

---
## Step 1 — Load and explore the data

In [ ]:
df = pd.read_csv(RAW_DATA)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print(f'Patients          : {len(df)}')
print(f'Deterioration rate: {df["deteriorated"].mean():.1%}')
print(f'Notes available   : {df["has_notes"].mean():.1%}')
print(f'Age mean / std    : {df["age"].mean():.1f} / {df["age"].std():.1f}')
print()
print('Missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

df['age'].hist(ax=axes[0], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Age distribution'); axes[0].set_xlabel('Age')

df.groupby('admission_type')['deteriorated'].mean().plot(
    kind='bar', ax=axes[1], color='tomato', edgecolor='white'
)
axes[1].set_title('Deterioration rate by admission type')
axes[1].set_ylabel('Rate'); axes[1].tick_params(axis='x', rotation=20)

df.groupby('has_notes')['deteriorated'].mean().plot(
    kind='bar', ax=axes[2], color=['tomato', 'seagreen'], edgecolor='white'
)
axes[2].set_title('Deterioration rate vs notes')
axes[2].set_xticklabels(['No notes', 'Has notes'], rotation=0)
plt.tight_layout(); plt.show()

### 1a — Missingness analysis

Understanding *why* values are missing matters before imputing.
Missingness correlated with the outcome (MNAR) biases any imputation strategy.
This is exactly the flaw built into Model C — investigate it here first.

In [ ]:
df['_age_band'] = pd.cut(
    df['age'], bins=[0, 64, 74, 200],
    labels=['under_65', '65_to_74', '75_plus']
)

note_by_adm  = df.groupby('admission_type')['note_risk_score'].apply(
    lambda x: x.isnull().mean()).rename('missing_rate')
note_by_age  = df.groupby('_age_band')['note_risk_score'].apply(
    lambda x: x.isnull().mean()).rename('missing_rate')
det_by_notes = df.groupby('has_notes')['deteriorated'].mean()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

note_by_adm.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Note missingness\nby admission type')
axes[0].set_ylabel('Missing rate'); axes[0].set_ylim(0, 0.8)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[0].tick_params(axis='x', rotation=20)

note_by_age.plot(kind='bar', ax=axes[1], color='seagreen', edgecolor='white')
axes[1].set_title('Note missingness\nby age band')
axes[1].set_ylabel('Missing rate'); axes[1].set_ylim(0, 0.8)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].tick_params(axis='x', rotation=20)

det_by_notes.index = ['No notes', 'Has notes']
det_by_notes.plot(kind='bar', ax=axes[2],
                  color=['tomato', 'seagreen'], edgecolor='white')
axes[2].set_title('Deterioration rate\nvs notes availability')
axes[2].set_ylabel('Deterioration rate'); axes[2].set_ylim(0, 0.6)
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[2].tick_params(axis='x', rotation=0)

plt.suptitle('Missingness is not random', fontsize=12)
plt.tight_layout(); plt.show()
df.drop(columns=['_age_band'], inplace=True)

print('Key finding:')
print(f'  Surgical patients missing notes : {note_by_adm["surgical"]:.1%}')
print(f'  Medical  patients missing notes : {note_by_adm["medical"]:.1%}')
print(f'  Deterioration WITHOUT notes     : {det_by_notes["No notes"]:.1%}')
print(f'  Deterioration WITH notes        : {det_by_notes["Has notes"]:.1%}')
print()
print('=> MNAR: missingness correlates with admission type and outcome.')
print('   Model C silently imputes the population mean — investigate the')
print('   consequences in Step 6b and the subgroup analysis below.')

---
## Step 2 — Preprocess

The model classes handle `note_risk_score` imputation internally.
Fill any remaining missing values in numeric columns.
The cell below compares three strategies so you can justify your choice.

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

numeric_cols = [
    'hr_mean', 'hr_std', 'rr_mean', 'rr_std',
    'spo2_mean', 'spo2_min', 'sbp_mean', 'temp_mean',
    'lactate', 'creatinine', 'wbc', 'bilirubin',
]

# Benchmark on artificially masked lactate values
rng_imp = np.random.RandomState(99)
imp_mask = rng_imp.rand(len(df)) < 0.15
df_imp   = df[numeric_cols].copy()
true_lac = df_imp.loc[imp_mask, 'lactate'].copy()
df_imp.loc[imp_mask, 'lactate'] = np.nan

print('RMSE for held-out lactate values (lower = better):')
for name, imp in [
    ('Median', SimpleImputer(strategy='median')),
    ('Mean',   SimpleImputer(strategy='mean')),
    ('KNN-5',  KNNImputer(n_neighbors=5)),
]:
    imputed = imp.fit_transform(df_imp)[:, numeric_cols.index('lactate')]
    rmse = np.sqrt(np.mean((imputed[imp_mask] - true_lac.values) ** 2))
    print(f'  {name:8s}: RMSE = {rmse:.4f}')

# Apply median imputation (robust, interpretable, no data leakage risk)
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

print(f'\nRemaining missing (excl. note_risk_score): '
      f'{df.drop(columns=["note_risk_score"]).isnull().sum().sum()} cells')

---
## Step 3 — Train / Validation / Test split

Set your preferred split ratios below. The split is stratified to preserve
class balance across all three sets. Ensure at least ~50 patients in val and test.

In [ ]:
from sklearn.model_selection import train_test_split

# ── TODO: adjust to your preferred ratios (must sum to 1.0) ─────────────
TRAIN_FRAC = 0   # proportion for training  — e.g. 0.60, 0.70, 0.75
VAL_FRAC   = 0   # proportion for validation — e.g. 0.15, 0.20
SEED       = 42     # random seed — change for a different split — e.g., 
# ────────────────────────────────────────────────────────────────────────

df_train, df_temp = train_test_split(
    df, test_size=1 - TRAIN_FRAC,
    stratify=df['deteriorated'], random_state=SEED
)
val_share = VAL_FRAC / (1 - TRAIN_FRAC)
df_val, df_test = train_test_split(
    df_temp, test_size=1 - val_share,
    stratify=df_temp['deteriorated'], random_state=SEED
)

for split, name in [(df_train,'train'), (df_val,'val'), (df_test,'test')]:
    print(f'{name:6s}: n={len(split):4d}  '
          f'deterioration={split["deteriorated"].mean():.1%}  '
          f'notes={split["has_notes"].mean():.1%}')

df_train.to_csv(PROC_DIR / 'train.csv', index=False)
df_val.to_csv(  PROC_DIR / 'val.csv',   index=False)
df_test.to_csv( PROC_DIR / 'test.csv',  index=False)
print('Splits saved to data/processed/')

---
## Step 4 — Define features and labels

In [ ]:
X_train = df_train.drop(columns=['patient_id', 'deteriorated'])
y_train = df_train['deteriorated'].values
X_val   = df_val.drop(columns=['patient_id', 'deteriorated'])
y_val   = df_val['deteriorated'].values
X_test  = df_test.drop(columns=['patient_id', 'deteriorated'])
y_test  = df_test['deteriorated'].values
print(f'X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')

---
## Step 5 — Train Model A *(LightGBM multimodal fusion — provided complete)*

**Model A** is a [LightGBM](https://lightgbm.readthedocs.io/) gradient boosting
classifier trained on all available modalities: demographics, SOFA severity score,
vital signs, laboratory results, and clinical notes. It is the strongest of the three
models and serves as your **reference baseline** — compare all findings against it.

> **Requires:** `src/evaluate.py` implemented. If `compute_metrics()` raises
> `NotImplementedError`, complete `src/evaluate.py` first then return here.

In [ ]:
model_a = ModelA()
model_a.fit(X_train, y_train)

m_a_train = compute_metrics(y_train, model_a.predict_proba(X_train)[:, 1], model_name='Model A')
m_a_val   = compute_metrics(y_val,   model_a.predict_proba(X_val)[:, 1],   model_name='Model A')
print(f'Model A | train AUROC={m_a_train["auroc"]:.3f}  val AUROC={m_a_val["auroc"]:.3f}')

fi_a = model_a.feature_importance().head(15)
fig, ax = plt.subplots(figsize=(8, 5))
fi_a.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Model A — LightGBM feature importance')
ax.set_xlabel('Importance score')
plt.tight_layout(); plt.show()

print('Top 5 features:')
for feat, val in fi_a.head(5).items():
    print(f'  {feat:<28} {val:.0f}')
print()
print('Clinical check: sofa_score, lactate, note_risk_imputed dominating is')
print('clinically sensible — severity score and perfusion marker are the')
print('strongest signals of 24-hour deterioration.')

---
## Step 6 — Train Model B and Model C *(implement these in models/ first)*

Both models are provided as skeletons. Read the design brief at the top of each
file before implementing. The brief explains the clinical context, the known
limitation, and the architectural options available to you.

The two model implementations are independent — each person on the team leads one model depending on your arrangement. Person 2 works on `models/model_b.py`, Person 3 works on `models/model_c.py`. Once both are implemented, run all cells in this section.

### 6a — Model B: Vitals-only Logistic Regression

In [ ]:
model_b = ModelB()
model_b.fit(X_train, y_train)

m_b_train = compute_metrics(y_train, model_b.predict_proba(X_train)[:, 1], model_name='Model B')
m_b_val   = compute_metrics(y_val,   model_b.predict_proba(X_val)[:, 1],   model_name='Model B')
print(f'Model B | train AUROC={m_b_train["auroc"]:.3f}  val AUROC={m_b_val["auroc"]:.3f}')

fi_b = model_b.feature_importance()
fig, ax = plt.subplots(figsize=(7, 4))
fi_b.sort_values().plot(kind='barh', ax=ax, color='tomato', edgecolor='white')
ax.set_title('Model B — feature importance')
ax.set_xlabel('Importance')
plt.tight_layout(); plt.show()

print('Note: Model B AUROC may be near-random because vital signs alone')
print('carry limited signal without labs or notes. Investigate the age gap next.')

### 6b — Model C: Notes-dependent model

In [ ]:
model_c = ModelC()
model_c.fit(X_train, y_train)

m_c_train = compute_metrics(y_train, model_c.predict_proba(X_train)[:, 1], model_name='Model C')
m_c_val   = compute_metrics(y_val,   model_c.predict_proba(X_val)[:, 1],   model_name='Model C')
print(f'Model C | train AUROC={m_c_train["auroc"]:.3f}  val AUROC={m_c_val["auroc"]:.3f}')

# Immediate notes breakdown — the key failure mode for Model C
prob_c_val = model_c.predict_proba(X_val)[:, 1]
for has, label in [(1, 'With notes   '), (0, 'Without notes')]:
    mask = df_val['has_notes'].values == has
    if mask.sum() >= 5:
        m = compute_metrics(y_val[mask], prob_c_val[mask])
        print(f'  Model C {label}: AUROC={m["auroc"]:.3f}  n={m["n_total"]}')

### 6c — Calibration check

A well-calibrated model that says '70% risk' should be right 70% of the time.
Poor calibration means scores cannot be used directly to prioritise care.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, model, name, colour in [
    (axes[0], model_a, 'Model A', 'steelblue'),
    (axes[1], model_b, 'Model B', 'tomato'),
    (axes[2], model_c, 'Model C', 'seagreen'),
]:
    prob = model.predict_proba(X_val)[:, 1]
    frac, mean_pred, counts = compute_calibration(y_val, prob, n_bins=8)
    ax.plot([0, 1], [0, 1], '--', color='grey', linewidth=1)
    sc = ax.scatter(mean_pred, frac, c=counts, cmap='Blues', s=80, zorder=5)
    ax.plot(mean_pred, frac, color=colour, linewidth=1.5, alpha=0.7)
    plt.colorbar(sc, ax=ax, label='N patients')
    ax.set_title(f'{name} — calibration')
    ax.set_xlabel('Mean predicted prob')
    ax.set_ylabel('Observed deterioration rate')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.suptitle('Calibration (val set) — points on diagonal = well calibrated', fontsize=11)
plt.tight_layout(); plt.show()

print('Points above diagonal = model under-predicts (conservative)')
print('Points below diagonal = model over-predicts (overconfident)')
print('Model B scattered randomly = near-random predictions')

---
## Step 7 — Validation summary

In [ ]:
val_results = []
for model_obj, name in [
    (model_a, 'Model A'),
    (model_b, 'Model B'),
    (model_c, 'Model C'),
]:
    prob = model_obj.predict_proba(X_val)[:, 1]
    val_results.append(compute_metrics(y_val, prob, model_name=name))

summary_cols = ['model', 'auroc', 'auprc', 'sensitivity', 'specificity', 'brier_score']
pd.DataFrame(val_results)[summary_cols]

---
## Step 8 — Save models and metrics

In [ ]:
joblib.dump(model_a, MODEL_DIR / 'model_a.joblib')
joblib.dump(model_b, MODEL_DIR / 'model_b.joblib')
joblib.dump(model_c, MODEL_DIR / 'model_c.joblib')
print('Models saved to saved_models/')

save_metrics_csv(val_results, str(METRIC_DIR / 'training_metrics.csv'))
print('Metrics saved to saved_metrics/training_metrics.csv')

---
## Steps 1–8 complete. Launch the evaluation app.

```bash
cd ..          # back to clinical_strand/
streamlit run app.py
```

The app loads your saved models and test set automatically.
Tabs: Data Overview → Model Performance → Subgroup Analysis → Failure Analysis → Report Builder

---
## Further analysis — run before or alongside the app

The cells below are starting points for each track role.
Run them on the **validation set** to guide what you look for in the app.

### Failure Hunter — subgroup analysis across all models

In [ ]:
df_val_plot = df_val.copy()
df_val_plot['age_band']    = age_band(df_val_plot)
df_val_plot['notes_label'] = df_val_plot['has_notes'].map({0: 'No notes', 1: 'Has notes'})

for group_col, group_label in [
    ('age_band',    'Age band'),
    ('notes_label', 'Notes availability'),
    ('admission_type', 'Admission type'),
    ('sex',         'Sex'),
]:
    print(f'\n=== {group_label} ===')
    for model, name in [(model_a,'A'), (model_b,'B'), (model_c,'C')]:
        prob = model.predict_proba(X_val)[:, 1]
        sub  = compute_subgroup_metrics(df_val_plot, prob, y_val, group_col)
        if sub.empty:
            continue
        ref  = sub.loc[sub['auroc'].idxmax(), 'group_value']
        sub  = equity_gap(sub, reference_value=ref, metric='sensitivity')
        print(f'  Model {name}:')
        print(sub[['group_value', 'n_group', 'auroc', 'sensitivity', 'sensitivity_gap']]
              .to_string(index=False))

### Failure Hunter — ablation experiment (Model B)

In [ ]:
# Re-train Model B with one feature group removed.
# Compare AUROC for patients aged 75+ before and after.
# This tests whether the failure is tied to a specific feature or is structural.

from models.model_b import VITALS
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

age_val = df_val['age'].values
mask_75 = age_val >= 75

def ablation_auroc(features, label):
    pipe = Pipeline([
        ('sc',  StandardScaler()),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
    ])
    pipe.fit(X_train[features], y_train)
    prob = pipe.predict_proba(X_val[features])[:, 1]
    m_all = compute_metrics(y_val, prob)
    m_75  = compute_metrics(y_val[mask_75], prob[mask_75])
    print(f'  {label:<40} overall={m_all["auroc"]:.3f}  '
          f'75+ AUROC={m_75["auroc"]:.3f}')

print('Ablation results (logistic regression, no age attenuation):')
ablation_auroc(VITALS,                              'Baseline — all vitals')
ablation_auroc([f for f in VITALS if 'std' not in f], 'Remove variability features')
ablation_auroc([f for f in VITALS if 'spo2' not in f], 'Remove SpO2 features')
ablation_auroc(VITALS + ['age'],                    'Add age to vitals')
print()
print(f'Model A  75+ AUROC (ceiling): '
      f'{compute_metrics(y_val[mask_75], model_a.predict_proba(X_val)[:, 1][mask_75])["auroc"]:.3f}')

### Explainer — feature importance comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, model, name, colour in [
    (axes[0], model_a, 'Model A — LightGBM',          'steelblue'),
    (axes[1], model_b, 'Model B — Vitals LR',         'tomato'),
    (axes[2], model_c, 'Model C — Notes-dependent',   'seagreen'),
]:
    fi = model.feature_importance().head(10)
    fi.sort_values().plot(kind='barh', ax=ax, color=colour, edgecolor='white')
    ax.set_title(name)
    ax.set_xlabel('Importance')

plt.suptitle('Feature importance — clinical sense check', fontsize=12)
plt.tight_layout(); plt.show()

print('Clinical check questions:')
print('  - Are the top features clinically plausible predictors of deterioration?')
print('  - Does Model C list note_risk_score or has_notes as a top feature?')
print('  - Does Model A over-rely on any single feature that might not generalise?')